# klev stitch demo — vanilla vs stitched, with latency

Loads **klev** (`/mnt/f/distill_jev_runs/main`, the model formerly called M5) plus the
external CoSt-BR LoRA (`Trained Models/My Dataset/gemma-4-e4b_claim`) at weight 0.5, builds a
**27-example probe in the LoRA's own prompt space**, and compares per-example decisions:

- **vanilla** — klev pointer head on the system-one prompt
- **stitch** — `logits_ptr + beta * log p_alp` (two scalars, no weight changes)

Reference full-test numbers: vanilla 0.405 → stitch 0.521 on the 718-row CoSt-BR bench
(`docs/m9-lora-stitch.md`).

In [1]:
import os, sys, time, json, importlib.util, statistics
from pathlib import Path

os.environ.setdefault("HF_HOME", "/mnt/f/huggingface/")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
REPO = Path.cwd()
if not (REPO / "model").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import torch
import pandas as pd

spec = importlib.util.spec_from_file_location("stitch", REPO / "eval" / "eval_lora_stitch.py")
stitch = importlib.util.module_from_spec(spec)
sys.modules["stitch"] = stitch
spec.loader.exec_module(stitch)
print(REPO)

[unsloth.import_fixes|WARNING]Unsloth: Detected broken vLLM binary extension; disabling vLLM imports and continuing import.
Please reinstall via `uv pip install unsloth vllm torchvision torchaudio --torch-backend=auto`.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


/home/penhfel/github/klev


In [2]:
from types import SimpleNamespace
a = SimpleNamespace(model=stitch.MODEL, run="/mnt/f/distill_jev_runs/main",
                    ext="/home/penhfel/github/Trained Models/My Dataset/gemma-4-e4b_claim",
                    weight=0.5)
model, tokenizer, head = stitch.load_model(a)
device = next(model.parameters()).device

from datasets import load_from_disk
train_ds = load_from_disk("/mnt/f/distill_jev_runs/prep/costbr-30")
test_ds = load_from_disk("/mnt/f/distill_jev_runs/prep/bench-costbr-test")
KEYS = test_ds[0]["keys"]
raw = {json.loads(l)["_meta"]["id"]: json.loads(l) for l in open("/mnt/f/distill_jev_runs/bench/costbr-test.jsonl")}
print("klev + LoRA loaded |", len(train_ds), "probe rows |", len(test_ds), "test rows")

==((====))==  Unsloth 2026.9.7: Fast Gemma4 patching. Transformers: 5.5.0. vLLM: 0.19.1.
   \\   /|    NVIDIA GeForce RTX 4080. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

klev + LoRA loaded | 27 probe rows | 718 test rows


In [3]:
# probe: class means in the LoRA's alpaca-prompt space (27 examples)
h_alp_tr = stitch.alpaca_features(model, tokenizer, stitch.alpaca_texts(train_ds, stitch.TRAIN_PROMPTS), device)
y_tr = torch.tensor([int(train_ds[i]["label"]) for i in range(len(train_ds))])
means = torch.stack([h_alp_tr[y_tr == c].mean(0) for c in range(len(KEYS))])
med = float(((torch.cdist(h_alp_tr, means)) ** 2).median())
T, beta = med * 0.5, 4.0
print(f"probe built: T={T:.1f} (median d^2={med:.1f}), beta={beta}")

/home/penhfel/unsloth_uv/lib/python3.13/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/home/penhfel/unsloth_uv/lib/python3.13/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/home/penhfel/unsloth_uv/lib/python3.13/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/home/penhfel/unsloth_uv/lib/python3.13/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


probe built: T=344.5 (median d^2=689.1), beta=4.0


In [4]:
# demo rows: spread across the test set (same records as the full eval)
demo_idx = [0, 50, 150, 250, 350, 450, 550, 650]
demo_ds = test_ds.select(demo_idx)
demo_texts = stitch.alpaca_texts(demo_ds, stitch.EVAL_PROMPTS)

h_sys, h_opts, owner, y_demo = stitch.sys_features(model, demo_ds, device)
h_alp_de = stitch.alpaca_features(model, tokenizer, demo_texts, device)
logits_ptr = stitch.head_option_logits(head, h_sys, h_opts, owner).reshape(len(demo_ds), -1)
p_alp = torch.softmax(-(((torch.cdist(h_alp_de, means)) ** 2)) / T, -1)
p_ptr = torch.softmax(logits_ptr, -1)
p_stitch = torch.softmax(torch.log(p_ptr + 1e-9) + beta * torch.log(p_alp + 1e-9), -1)

rows = []
for i in range(len(demo_ds)):
    rec = raw[demo_ds[i]["record_id"]]
    gold = KEYS[int(y_demo[i])]
    v, s = int(p_ptr[i].argmax()), int(p_stitch[i].argmax())
    rows.append({"message": rec["state"]["current_message"][:70], "gold": gold,
                 "vanilla": f"{KEYS[v]} ({float(p_ptr[i][v]):.2f})",
                 "stitch": f"{KEYS[s]} ({float(p_stitch[i][s]):.2f})",
                 "vanilla_ok": v == int(y_demo[i]), "stitch_ok": s == int(y_demo[i]),
                 "fixed": s == int(y_demo[i]) and v != int(y_demo[i])})
df = pd.DataFrame(rows)
print(f"demo accuracy: vanilla {df.vanilla_ok.mean():.0%} | stitch {df.stitch_ok.mean():.0%} | flipped to correct: {df.fixed.sum()}")
df

demo accuracy: vanilla 50% | stitch 75% | flipped to correct: 2


,message,gold,vanilla,stitch,vanilla_ok,stitch_ok,fixed
0,STF legislando já nem é surpreendente hoje em dia,concorda,concorda (0.91),concorda (0.99),True,True,False
1,[removed],discorda,irrelevante (0.77),discorda (0.95),False,True,True
2,Pelo menos algo de bom saindo do STF,concorda,concorda (0.93),concorda (0.99),True,True,False
3,"Concordo com você na conclusão, de que o Lula ...",discute,irrelevante (0.61),concorda (0.62),False,False,False
4,Como consegue defender uma medida arrombada de...,concorda,concorda (0.93),concorda (0.95),True,True,False
5,Pouquíssimas pessoas recebem todo o seu dinhei...,discorda,discorda (0.84),discorda (0.84),True,True,False
6,😂,concorda,irrelevante (0.76),irrelevante (1.00),False,False,False
7,"Então aguenta a punição quando for pêgo, ué.\n...",concorda,irrelevante (0.36),concorda (0.76),False,True,True


In [5]:
import time

@torch.no_grad()
def vanilla_step(row):
    ids = torch.tensor([row["input_ids"]], device=device)
    hidden = model(input_ids=ids, output_hidden_states=True, use_cache=False).hidden_states[-1][0]
    d = hidden[row["decide_pos"]].float()
    o = hidden[torch.tensor(row["opt_pos"], device=device)].float()
    z = head.many(d.unsqueeze(0), o, torch.zeros(len(row["opt_pos"]), dtype=torch.long, device=device))
    return torch.softmax(z.float(), -1).cpu()

@torch.no_grad()
def alpaca_step(text):
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=True)["input_ids"].to(device)
    h = model(input_ids=ids, output_hidden_states=True, use_cache=False).hidden_states[-1][0][-1].float().cpu()
    return torch.softmax(-(((h - means) ** 2).sum(-1)) / T, -1)

def stitch_step(row, text):
    p = vanilla_step(row)
    q = alpaca_step(text)
    return torch.softmax(torch.log(p + 1e-9) + beta * torch.log(q + 1e-9), -1)

for _ in range(3):                       # warmup (kernel autotune, cache)
    vanilla_step(demo_ds[0]); stitch_step(demo_ds[0], demo_texts[0])
torch.cuda.synchronize()

van, sti = [], []
for rep in range(15):
    row, text = demo_ds[rep % len(demo_ds)], demo_texts[rep % len(demo_ds)]
    torch.cuda.synchronize(); t0 = time.perf_counter(); vanilla_step(row); torch.cuda.synchronize()
    van.append((time.perf_counter() - t0) * 1000)
    torch.cuda.synchronize(); t0 = time.perf_counter(); stitch_step(row, text); torch.cuda.synchronize()
    sti.append((time.perf_counter() - t0) * 1000)

lat = {"vanilla_median_ms": round(statistics.median(van), 1), "stitch_median_ms": round(statistics.median(sti), 1),
       "vanilla_p90_ms": round(sorted(van)[int(0.9 * len(van)) - 1], 1), "stitch_p90_ms": round(sorted(sti)[int(0.9 * len(sti)) - 1], 1),
       "overhead_ms": round(statistics.median(sti) - statistics.median(van), 1),
       "overhead_ratio": round(statistics.median(sti) / statistics.median(van), 2)}
print(json.dumps(lat, indent=1))

{
 "vanilla_median_ms": 205.1,
 "stitch_median_ms": 456.4,
 "vanilla_p90_ms": 230.8,
 "stitch_p90_ms": 495.4,
 "overhead_ms": 251.3,
 "overhead_ratio": 2.23
}


**Read-out.** The stitch costs one extra forward over the LoRA's prompt (the system-one
decision forward is always needed), i.e. roughly +1 decision latency; in exchange it adds the
foreign LoRA's knowledge without touching a single klev weight — on the full bench,
0.405 → 0.521 (two scalars, 27 examples). Exact overhead on this machine is in the cell
above.

In [6]:
out = Path("/mnt/f/distill_jev_runs/stitch-demo"); out.mkdir(parents=True, exist_ok=True)
(out / "demo.json").write_text(json.dumps({"examples": rows, "latency": lat}, indent=1))
print("saved", out / "demo.json")

saved /mnt/f/distill_jev_runs/stitch-demo/demo.json
